# 03_Train.ipynb: Walk-Forward, Calibration & Model Training
Tuân thủ nghiêm ngặt AGENTS.md và PROCESS.md: Không rò rỉ dữ liệu, chỉ sử dụng tập development (df_dev), Walk-Forward 3 outer validation folds, 14 ngày calibration lịch sử, adaptive capacity threshold tại từng origin qua capacity_threshold_at_origin, đóng băng mô hình, xác lập evaluation span holdout thuần túy từ lịch calendar và xuất experiment.joblib.

In [1]:
%run 02_Preprocessing.ipynb

import hashlib
import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, ExtraTreesRegressor
from sklearn.feature_selection import f_regression
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, mean_absolute_error

# 1. Load data strictly on development cutoff
dev_cutoff = pd.Timestamp('2008-10-02 20:54:00')
df_raw = load_secom()
df_dev = df_raw[df_raw['timestamp'] < dev_cutoff].copy()
assert (df_dev['timestamp'] < dev_cutoff).all(), 'Holdout leakage in df_dev!'

# Configurable forecast horizon parameter (HOURS_H)
HOURS_H = 24
HORIZON_DELTA = pd.Timedelta(hours=HOURS_H)

# Build dataset with causal history features & 72h sensor means
X_dev, meta_dev = build_forecast_dataset(
    df_dev,
    lead=pd.Timedelta(hours=6),
    horizon=HORIZON_DELTA,
    history=pd.Timedelta(hours=168),
    delay=pd.Timedelta(hours=6),
    cadence=pd.Timedelta(hours=6),
    q=0.10,
    min_event_samples=5,
    include_sensor_means=True
)
print(f"X_dev shape: {X_dev.shape}, meta_dev shape: {meta_dev.shape}")
expected_origins = 269 if HOURS_H == 24 else (265 if HOURS_H == 48 else len(X_dev))
assert len(X_dev) == expected_origins, f"Expected {expected_origins} origins for H={HOURS_H}h, got {len(X_dev)}"
assert not X_dev.empty and len(X_dev) == len(meta_dev), "Integrity check failed on X_dev/meta_dev!"

All synthetic tests and causal checks passed.
df_dev shape: (1253, 593)
X_dev shape: (269, 15), meta_dev shape: (269, 9)
Event label counts:
event_label
0.0    157
1.0     72
NaN     40
Name: count, dtype: int64
Maturity mask sum at fit_time: 17 / 269
capacity_threshold_at_origin future-invariance test passed.


X_dev shape: (269, 605), meta_dev shape: (269, 9)


In [2]:
def tune_initial_capacity_threshold(scores, calib_meta, calib_raw, target_workloads=[0.20, 0.25, 0.30], min_lead=pd.Timedelta(hours=6)):
    """Compute static capacity-based initial fallback thresholds ignoring non-finite scores."""
    sentinel = float('inf')
    valid_scores = scores[np.isfinite(scores)]
    if len(valid_scores) == 0:
        return {b: sentinel for b in target_workloads}
        
    q_grid = np.linspace(0.0, 1.0, 21)
    cand_thresholds = np.unique(np.quantile(valid_scores, q_grid))
    cand_thresholds = np.sort(np.append(cand_thresholds, sentinel))
    
    inc = compute_incidence_matrix(calib_raw, calib_meta, min_lead=min_lead)
    total_n = len(calib_raw)
    
    best_taus = {}
    for budget in target_workloads:
        best_tau = sentinel
        best_wl = -1.0
        
        for tau in cand_thresholds:
            warn_mask = (scores >= tau)
            if not warn_mask.any():
                wl = 0.0
            else:
                rec_cov = inc[warn_mask].any(axis=0)
                wl = float(rec_cov.sum() / total_n) if total_n > 0 else 0.0
            
            if wl <= budget:
                if (wl > best_wl) or (np.isclose(wl, best_wl) and tau > best_tau):
                    best_wl = wl
                    best_tau = tau
        best_taus[budget] = float(best_tau)
    return best_taus

In [3]:
history_cols = [
    'n_avail_24h', 'n_avail_72h', 'n_avail_168h',
    'fail_rate_24h', 'fail_rate_72h', 'fail_rate_168h',
    'missingness_24h', 'missingness_72h', 'missingness_168h',
    'fail_rate_trend_24_vs_168', 'hours_since_last_avail'
]
calendar_cols = [
    'sin_hour_target_start', 'cos_hour_target_start',
    'sin_dow_target_start', 'cos_dow_target_start'
]
sensor_cols = [c for c in X_dev.columns if c.startswith('sensor_')]

def get_candidate_specs():
    return {
        'never_alert': {'type': 'never_alert'},
        'baseline_rate24h': {'type': 'baseline', 'col': 'fail_rate_24h'},
        'baseline_rate72h': {'type': 'baseline', 'col': 'fail_rate_72h'},
        'baseline_rate168h': {'type': 'baseline', 'col': 'fail_rate_168h'},
        'lr_history_c01': {
            'type': 'pipeline', 'features': history_cols, 'target': 'event',
            'model': Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(C=0.1, class_weight='balanced', random_state=42))
            ])
        },
        'lr_history_c10': {
            'type': 'pipeline', 'features': history_cols, 'target': 'event',
            'model': Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(C=1.0, class_weight='balanced', random_state=42))
            ])
        },
        'ridge_history_a10': {
            'type': 'pipeline', 'features': history_cols, 'target': 'rate',
            'model': Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('reg', Ridge(alpha=10.0, random_state=42))
            ])
        },
        'lr_sensors_k8': {
            'type': 'pipeline', 'features': history_cols + sensor_cols, 'target': 'event',
            'model': Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('selector', SelectKBest(score_func=f_classif, k=8)),
                ('clf', LogisticRegression(C=0.1, class_weight='balanced', random_state=42))
            ])
        },
        'rf_history': {
            'type': 'pipeline', 'features': history_cols, 'target': 'event',
            'model': Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('clf', RandomForestClassifier(
                    n_estimators=100, max_depth=3, min_samples_leaf=10, random_state=42
                ))
            ])
        },
        'lr_history_calendar_c01': {
            'type': 'pipeline', 'features': history_cols + calendar_cols, 'target': 'event',
            'model': Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(C=0.1, class_weight='balanced', random_state=42))
            ])
        },
        'rf_reg_history_calendar': {
            'type': 'pipeline', 'features': history_cols + calendar_cols, 'target': 'rate',
            'model': Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('reg', RandomForestRegressor(
                    n_estimators=150, max_depth=4, min_samples_leaf=8, random_state=42, n_jobs=2
                ))
            ])
        },
        'et_reg_history_calendar_sensors_k12': {
            'type': 'pipeline', 'features': history_cols + calendar_cols + sensor_cols, 'target': 'rate',
            'model': Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('selector', SelectKBest(score_func=f_regression, k=12)),
                ('reg', ExtraTreesRegressor(
                    n_estimators=150, max_depth=4, min_samples_leaf=8, random_state=42, n_jobs=2
                ))
            ])
        }
    }


In [4]:
val_end = meta_dev['target_end'].max()
val_spans = []
for fold_idx in range(3):
    e_end = val_end - pd.Timedelta(days=14 * (2 - fold_idx))
    e_start = e_end - pd.Timedelta(days=14)
    val_spans.append((e_start, e_end))

print('Outer validation target spans:')
for idx, (s, e) in enumerate(val_spans):
    print(f'  Fold {idx+1}: {s} to {e}')

budgets = [0.20, 0.25, 0.30]
fold_records = []
diagnostics = {}

for fold_idx, (eval_start, eval_end) in enumerate(val_spans):
    val_mask = (meta_dev['target_start'] >= eval_start) & (meta_dev['target_end'] <= eval_end)
    val_meta = meta_dev[val_mask]
    first_val_origin = val_meta.index.min()
    
    calib_start = first_val_origin - pd.Timedelta(days=14)
    calib_mask = (meta_dev.index >= calib_start) & (meta_dev.index < first_val_origin) & (meta_dev['target_available'] <= first_val_origin)
    calib_meta = meta_dev[calib_mask]
    first_calib_origin = calib_meta.index.min()
    
    assert calib_meta['target_available'].max() <= first_val_origin, 'Calibration leakage into validation!'
    
    core_train_mask = training_mask(meta_dev, fit_time=first_calib_origin)
    core_meta = meta_dev[core_train_mask]
    core_X = X_dev[core_train_mask]
    assert core_meta['target_available'].max() <= first_calib_origin, 'Core training leakage into calibration!'
    
    train_event_mask = ~core_meta['event_label'].isna()
    train_rate_mask = ~core_meta['rate'].isna()
    val_event_mask = ~val_meta['event_label'].isna()
    
    val_raw = df_dev[(df_dev['timestamp'] >= eval_start) & (df_dev['timestamp'] < eval_end)].copy()
    calib_raw = df_dev[(df_dev['timestamp'] >= calib_meta['target_start'].min()) & (df_dev['timestamp'] < calib_meta['target_end'].max())].copy()
    
    # Combined calibration + validation contexts for prequential online policy scoring
    # Combined calibration + validation contexts: include EVERY calendar origin from first_calib_origin to last validation origin
    last_val_origin = val_meta.index.max()
    eval_comb_mask = (meta_dev.index >= first_calib_origin) & (meta_dev.index <= last_val_origin)
    eval_comb_meta = meta_dev[eval_comb_mask]
    
    specs = get_candidate_specs()
    diagnostics[f'fold_{fold_idx+1}'] = {}
    
    for cand_name, spec in specs.items():
        train_ap = np.nan
        val_ap = np.nan
        train_mae = np.nan
        val_mae = np.nan
        skip_reason = None
        
        if spec['type'] == 'never_alert':
            comb_scores = pd.Series(-np.inf, index=eval_comb_meta.index)
            initial_taus = {b: float('inf') for b in budgets}
        elif spec['type'] == 'baseline':
            col = spec['col']
            comb_scores = X_dev.loc[eval_comb_meta.index, col].fillna(0.0)
            val_scores_only = comb_scores.loc[val_meta.index].values
            if val_event_mask.sum() > 0 and len(np.unique(val_meta.loc[val_event_mask, 'event_label'])) > 1:
                val_ap = float(average_precision_score(val_meta.loc[val_event_mask, 'event_label'], val_scores_only[val_event_mask]))
            calib_scores_only = comb_scores.loc[calib_meta.index].values
            initial_taus = tune_initial_capacity_threshold(calib_scores_only, calib_meta, calib_raw, target_workloads=budgets)
        elif spec['type'] == 'pipeline':
            pipe = spec['model']
            feats = spec['features']
            if spec['target'] == 'event':
                y_train = core_meta.loc[train_event_mask, 'event_label'].values
                X_tr = core_X.loc[train_event_mask, feats]
                if len(np.unique(y_train)) < 2:
                    prior_val = float(y_train[0]) if len(y_train) > 0 else 0.0
                    comb_scores = pd.Series(prior_val, index=eval_comb_meta.index)
                    skip_reason = 'single_class_train'
                else:
                    pipe.fit(X_tr, y_train)
                    p_train = pipe.predict_proba(X_tr)[:, 1]
                    if len(np.unique(y_train)) > 1:
                        train_ap = float(average_precision_score(y_train, p_train))
                    p_comb = pipe.predict_proba(X_dev.loc[eval_comb_meta.index, feats])[:, 1]
                    comb_scores = pd.Series(p_comb, index=eval_comb_meta.index)
                    val_scores_only = comb_scores.loc[val_meta.index].values
                    if val_event_mask.sum() > 0 and len(np.unique(val_meta.loc[val_event_mask, 'event_label'])) > 1:
                        val_ap = float(average_precision_score(val_meta.loc[val_event_mask, 'event_label'], val_scores_only[val_event_mask]))
            else:
                y_train = core_meta.loc[train_rate_mask, 'rate'].values
                X_tr = core_X.loc[train_rate_mask, feats]
                if len(y_train) == 0:
                    comb_scores = pd.Series(0.0, index=eval_comb_meta.index)
                    skip_reason = 'empty_train_rate'
                else:
                    pipe.fit(X_tr, y_train)
                    p_train_rate = np.clip(pipe.predict(X_tr), 0.0, 1.0)
                    train_mae = float(mean_absolute_error(y_train, p_train_rate))
                    p_comb = np.clip(pipe.predict(X_dev.loc[eval_comb_meta.index, feats]), 0.0, 1.0)
                    comb_scores = pd.Series(p_comb, index=eval_comb_meta.index)
                    val_scores_only = comb_scores.loc[val_meta.index].values
                    val_rate_mask = ~val_meta['rate'].isna()
                    if val_rate_mask.sum() > 0:
                        val_mae = float(mean_absolute_error(val_meta.loc[val_rate_mask, 'rate'], val_scores_only[val_rate_mask]))
                    if val_event_mask.sum() > 0 and len(np.unique(val_meta.loc[val_event_mask, 'event_label'])) > 1:
                        val_ap = float(average_precision_score(val_meta.loc[val_event_mask, 'event_label'], val_scores_only[val_event_mask]))
            
            calib_scores_only = comb_scores.loc[calib_meta.index].values
            initial_taus = tune_initial_capacity_threshold(calib_scores_only, calib_meta, calib_raw, target_workloads=budgets)
        
        diagnostics[f'fold_{fold_idx+1}'][cand_name] = {
            'train_ap': train_ap,
            'val_ap': val_ap,
            'train_mae': train_mae,
            'val_mae': val_mae,
            'skip_reason': skip_reason,
            'initial_taus': initial_taus,
            'tau_audits': {}
        }
        
        # Prequential online policy: evaluate capacity threshold at EACH validation origin
        for b in budgets:
            tau_arr = []
            warn_v = []
            tau_audits_list = []
            
            for orig in val_meta.index:
                tau_t, audit_t = capacity_threshold_at_origin(
                    comb_scores,
                    eval_comb_meta,
                    df_dev,
                    current_origin=orig,
                    model_ready_time=first_calib_origin,
                    budget=b,
                    lookback=pd.Timedelta(days=14),
                    min_lead=pd.Timedelta(hours=6),
                    n_quantiles=21,
                    frozen_initial_threshold=initial_taus[b]
                )
                score_t = comb_scores.loc[orig]
                warn_t = bool(score_t >= tau_t)
                tau_arr.append(tau_t)
                warn_v.append(warn_t)
                tau_audits_list.append(audit_t)
                
            warn_v = np.array(warn_v, dtype=bool)
            tau_arr = np.array(tau_arr, dtype=float)
            
            tau_min = float(np.min(tau_arr))
            tau_max = float(np.max(tau_arr))
            tau_mean = float(np.mean(tau_arr))
            
            diagnostics[f'fold_{fold_idx+1}'][cand_name]['tau_audits'][b] = {
                'tau_min': tau_min,
                'tau_max': tau_max,
                'tau_mean': tau_mean,
                'first_tau': float(tau_arr[0]),
                'last_tau': float(tau_arr[-1])
            }
            
            m_v = interval_metrics(val_raw, val_meta, warn_v, eval_start, eval_end, min_lead=pd.Timedelta(hours=6))
            fold_records.append({
                'fold': fold_idx + 1,
                'candidate': cand_name,
                'budget': b,
                'tau_mean': tau_mean,
                'tau_min': tau_min,
                'tau_max': tau_max,
                'train_ap': train_ap,
                'val_ap': val_ap,
                'train_mae': train_mae,
                'val_mae': val_mae,
                'covered_fail': m_v['covered_fail'],
                'total_fail': m_v['total_fail'],
                'coverage': m_v['coverage'],
                'covered_n': m_v['covered_n'],
                'total_n': m_v['total_n'],
                'workload': m_v['workload'],
                'time_under_alert': m_v['time_under_alert'],
                'event_precision': m_v['event_precision'],
                'event_recall': m_v['event_recall']
            })

df_folds = pd.DataFrame(fold_records)
print('Walk-Forward validation completed across 3 folds with adaptive capacity thresholds.')

Outer validation target spans:
  Fold 1: 2008-08-21 18:00:00 to 2008-09-04 18:00:00
  Fold 2: 2008-09-04 18:00:00 to 2008-09-18 18:00:00
  Fold 3: 2008-09-18 18:00:00 to 2008-10-02 18:00:00


D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['sensor_85_mean72h' 'sensor_109_mean72h' 'sensor_110_mean72h'
 'sensor_111_mean72h' 'sensor_220_mean72h' 'sensor_244_mean72h'
 'sensor_245_mean72h' 'sensor_246_mean72h' 'sensor_358_mean72h'
 'sensor_382_mean72h' 'sensor_383_mean72h' 'sensor_384_mean72h'
 'sensor_492_mean72h' 'sensor_516_mean72h' 'sensor_517_mean72h'
 'sensor_518_mean72h']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [ 16  24  53  60  63  80  85 107 148 156 185 186 193 196 197 198 199 200
 201 213 216 232 235 236 237 238 239 240 241 242 243 246 247 248 249 259
 260 261 262 263 264 265 266 267 268 269 279 287 316 317 318 325 328 329
 330 331 332 333 345 350 366 371 372 373

D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['sensor_85_mean72h' 'sensor_109_mean72h' 'sensor_110_mean72h'
 'sensor_111_mean72h' 'sensor_220_mean72h' 'sensor_244_mean72h'
 'sensor_245_mean72h' 'sensor_246_mean72h' 'sensor_358_mean72h'
 'sensor_382_mean72h' 'sensor_383_mean72h' 'sensor_384_mean72h'
 'sensor_492_mean72h' 'sensor_516_mean72h' 'sensor_517_mean72h'
 'sensor_518_mean72h']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['sensor_85_mean72h' 'sensor_109_mean72h' 'sensor_110_mean72h'
 'sensor_111_mean72h' 'sensor_220_mean72h' 'sensor_244_mean72h'
 'sensor_245_mean72h' 'sensor_246_mean72h' 'sensor_358_mean72h'
 'sensor_382_mean72h' 'sensor_383_mean72h' 'sensor_384_me

D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['sensor_85_mean72h' 'sensor_109_mean72h' 'sensor_110_mean72h'
 'sensor_111_mean72h' 'sensor_220_mean72h' 'sensor_244_mean72h'
 'sensor_245_mean72h' 'sensor_246_mean72h' 'sensor_358_mean72h'
 'sensor_382_mean72h' 'sensor_383_mean72h' 'sensor_384_mean72h'
 'sensor_492_mean72h' 'sensor_516_mean72h' 'sensor_517_mean72h'
 'sensor_518_mean72h']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [ 16  24  53  60  63  80  85 107 148 156 185 186 193 196 197 198 199 200
 201 213 216 232 235 236 237 238 239 240 241 242 243 246 247 248 249 259
 260 261 262 263 264 265 266 267 268 269 279 287 316 317 318 325 328 329
 330 331 332 333 345 350 366 371 372 373

D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['sensor_85_mean72h' 'sensor_109_mean72h' 'sensor_110_mean72h'
 'sensor_111_mean72h' 'sensor_220_mean72h' 'sensor_244_mean72h'
 'sensor_245_mean72h' 'sensor_246_mean72h' 'sensor_358_mean72h'
 'sensor_382_mean72h' 'sensor_383_mean72h' 'sensor_384_mean72h'
 'sensor_492_mean72h' 'sensor_516_mean72h' 'sensor_517_mean72h'
 'sensor_518_mean72h']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['sensor_85_mean72h' 'sensor_109_mean72h' 'sensor_110_mean72h'
 'sensor_111_mean72h' 'sensor_220_mean72h' 'sensor_244_mean72h'
 'sensor_245_mean72h' 'sensor_246_mean72h' 'sensor_358_mean72h'
 'sensor_382_mean72h' 'sensor_383_mean72h' 'sensor_384_me

D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['sensor_85_mean72h' 'sensor_109_mean72h' 'sensor_110_mean72h'
 'sensor_111_mean72h' 'sensor_220_mean72h' 'sensor_244_mean72h'
 'sensor_245_mean72h' 'sensor_246_mean72h' 'sensor_358_mean72h'
 'sensor_382_mean72h' 'sensor_383_mean72h' 'sensor_384_mean72h'
 'sensor_492_mean72h' 'sensor_516_mean72h' 'sensor_517_mean72h'
 'sensor_518_mean72h']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [ 16  24  53  60  63  80  85 107 148 156 185 186 193 196 197 198 199 200
 201 213 216 232 235 236 237 238 239 240 241 242 243 246 247 248 249 259
 260 261 262 263 264 265 266 267 268 269 279 287 316 317 318 325 328 329
 330 331 332 333 345 350 366 371 372 373

D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['sensor_85_mean72h' 'sensor_109_mean72h' 'sensor_110_mean72h'
 'sensor_111_mean72h' 'sensor_220_mean72h' 'sensor_244_mean72h'
 'sensor_245_mean72h' 'sensor_246_mean72h' 'sensor_358_mean72h'
 'sensor_382_mean72h' 'sensor_383_mean72h' 'sensor_384_mean72h'
 'sensor_492_mean72h' 'sensor_516_mean72h' 'sensor_517_mean72h'
 'sensor_518_mean72h']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['sensor_85_mean72h' 'sensor_109_mean72h' 'sensor_110_mean72h'
 'sensor_111_mean72h' 'sensor_220_mean72h' 'sensor_244_mean72h'
 'sensor_245_mean72h' 'sensor_246_mean72h' 'sensor_358_mean72h'
 'sensor_382_mean72h' 'sensor_383_mean72h' 'sensor_384_me

Walk-Forward validation completed across 3 folds with adaptive capacity thresholds.


In [5]:
summary_table = df_folds.groupby(['candidate', 'budget']).agg({
    'covered_fail': 'sum',
    'total_fail': 'sum',
    'covered_n': 'sum',
    'total_n': 'sum'
}).reset_index()

summary_table['agg_coverage'] = np.where(summary_table['total_fail'] > 0, summary_table['covered_fail'] / summary_table['total_fail'], np.nan)
summary_table['agg_workload'] = np.where(summary_table['total_n'] > 0, summary_table['covered_n'] / summary_table['total_n'], 0.0)

print('=== SUMMARY TABLE (All Candidates & Budgets) ===')
display_cols = ['candidate', 'budget', 'covered_fail', 'total_fail', 'agg_coverage', 'covered_n', 'total_n', 'agg_workload']
print(summary_table[display_cols].to_string(index=False))

eligible = summary_table[summary_table['agg_workload'] <= 0.30].copy()
if not eligible.empty:
    eligible = eligible.sort_values(by=['agg_coverage', 'agg_workload'], ascending=[False, True])
    winner_row = eligible.iloc[0]
    winner_cand = winner_row['candidate']
    winner_budget = winner_row['budget']
    winner_cov = winner_row['agg_coverage']
    winner_wl = winner_row['agg_workload']
else:
    winner_cand = 'never_alert'
    winner_budget = 0.0
    winner_cov = 0.0
    winner_wl = 0.0

print(f'\n=== SELECTED WINNER (Development Only) ===')
print(f'Candidate: {winner_cand} | Budget Policy: {winner_budget}')
print(f'Agg Coverage: {winner_cov:.4f} | Agg Workload: {winner_wl:.4f}')
target_70_achieved = (winner_cov >= 0.70)
print(f'Mục tiêu 70% đạt được: {target_70_achieved}')

print(f'\n=== SELECTED CANDIDATE PER-FOLD RESULTS ===')
winner_folds = df_folds[(df_folds['candidate'] == winner_cand) & (df_folds['budget'] == winner_budget)]
print(winner_folds[['fold', 'tau_mean', 'tau_min', 'tau_max', 'covered_fail', 'total_fail', 'coverage', 'covered_n', 'total_n', 'workload', 'val_ap']].to_string(index=False))

dev_last_origin = meta_dev.index.max()
deploy_calib_start = dev_last_origin - pd.Timedelta(days=14)
deploy_calib_mask = (meta_dev.index >= deploy_calib_start) & (meta_dev['target_available'] <= dev_cutoff)
deploy_calib_meta = meta_dev[deploy_calib_mask]
first_deploy_calib_origin = deploy_calib_meta.index.min()
policy_ready_time = dev_cutoff

assert deploy_calib_meta['target_available'].max() <= policy_ready_time, 'Deploy calibration exceeds policy_ready_time!'

deploy_train_mask = training_mask(meta_dev, fit_time=first_deploy_calib_origin)
deploy_core_meta = meta_dev[deploy_train_mask]
deploy_core_X = X_dev[deploy_train_mask]
assert deploy_core_meta['target_available'].max() <= first_deploy_calib_origin, 'Deploy core training exceeds calibration start!'

calib_target_start = deploy_calib_meta['target_start'].min()
calib_target_end = deploy_calib_meta['target_end'].max()
deploy_calib_raw = df_dev[(df_dev['timestamp'] >= calib_target_start) & (df_dev['timestamp'] < calib_target_end)].copy()

deploy_specs = get_candidate_specs()
deploy_fitted_models = {}
deploy_initial_thresholds = {}

for c_name, c_spec in deploy_specs.items():
    if c_spec['type'] == 'never_alert':
        deploy_fitted_models[c_name] = {'type': 'never_alert'}
        deploy_initial_thresholds[c_name] = {b: float('inf') for b in budgets}
    elif c_spec['type'] == 'baseline':
        deploy_fitted_models[c_name] = {'type': 'baseline', 'col': c_spec['col']}
        c_calib_scores = X_dev.loc[deploy_calib_meta.index, c_spec['col']].fillna(0.0).values
        deploy_initial_thresholds[c_name] = tune_initial_capacity_threshold(c_calib_scores, deploy_calib_meta, deploy_calib_raw, target_workloads=budgets)
    elif c_spec['type'] == 'pipeline':
        p = c_spec['model']
        feats = c_spec['features']
        if c_spec['target'] == 'event':
            ev_mask = ~deploy_core_meta['event_label'].isna()
            y_tr = deploy_core_meta.loc[ev_mask, 'event_label'].values
            X_tr = deploy_core_X.loc[ev_mask, feats]
            if len(np.unique(y_tr)) < 2:
                deploy_fitted_models[c_name] = {'type': 'prior_fallback', 'value': float(y_tr[0]) if len(y_tr) > 0 else 0.0}
                c_calib_scores = np.full(len(deploy_calib_meta), deploy_fitted_models[c_name]['value'])
            else:
                p.fit(X_tr, y_tr)
                deploy_fitted_models[c_name] = p
                c_calib_scores = p.predict_proba(X_dev.loc[deploy_calib_meta.index, feats])[:, 1]
        else:
            rt_mask = ~deploy_core_meta['rate'].isna()
            y_tr = deploy_core_meta.loc[rt_mask, 'rate'].values
            X_tr = deploy_core_X.loc[rt_mask, feats]
            if len(y_tr) == 0:
                deploy_fitted_models[c_name] = {'type': 'prior_fallback', 'value': 0.0}
                c_calib_scores = np.zeros(len(deploy_calib_meta))
            else:
                p.fit(X_tr, y_tr)
                deploy_fitted_models[c_name] = p
                c_calib_scores = np.clip(p.predict(X_dev.loc[deploy_calib_meta.index, feats]), 0.0, 1.0)
            
        deploy_initial_thresholds[c_name] = tune_initial_capacity_threshold(c_calib_scores, deploy_calib_meta, deploy_calib_raw, target_workloads=budgets)

lead_delta = pd.Timedelta(hours=6)
horizon_delta = HORIZON_DELTA
cadence_delta = pd.Timedelta(hours=6)

earliest_holdout_origin = dev_cutoff.ceil(cadence_delta)
planned_eval_start = earliest_holdout_origin + lead_delta

max_raw_ts = df_raw['timestamp'].max()
latest_holdout_origin = (max_raw_ts - lead_delta - horizon_delta).floor(cadence_delta)
planned_eval_end = latest_holdout_origin + lead_delta + horizon_delta

print(f'\n=== FROZEN HOLDOUT EVALUATION PROTOCOL ===')
print(f'Earliest Holdout Origin: {earliest_holdout_origin}')
print(f'Planned Eval Start: {planned_eval_start}')
print(f'Planned Eval End: {planned_eval_end}')

hash_secom = '20f0e7ee434f7dcbae0eea9ffff009a2b57f42d6b0dc9a5bd4f00782c0a3374c'
hash_labels = '126884cf453705c9e61a903fe906f0665a3b45ce3639e621edc5c93c89627e03'

exp_artifact = {
    'protocol': {
        'lead': '6h', 'horizon': f'{HOURS_H}h', 'history': '168h', 'delay': '6h',
        'cadence': '6h', 'q': 0.10, 'min_event_samples': 5,
        'dev_cutoff': str(dev_cutoff),
        'holdout_known_exposure': '314 observations displayed in historical notebook; not blind'
    },
    'policy': {
        'kind': 'adaptive_capacity',
        'lookback': '14d',
        'n_quantiles': 21,
        'model_ready_time': str(first_deploy_calib_origin),
        'policy_ready_time': str(policy_ready_time),
        'decision_rule': 'online adaptive via capacity_threshold_at_origin; scalar is initial past-only fallback'
    },
    'file_hashes': {
        'secom_data_sha256': hash_secom,
        'secom_labels_sha256': hash_labels
    },
    'feature_columns': {
        'history_cols': history_cols,
        'calendar_cols': calendar_cols,
        'sensor_cols': sensor_cols
    },
    'temporal_bounds': {
        'first_deploy_calib_origin': str(first_deploy_calib_origin),
        'deploy_calib_start': str(deploy_calib_start),
        'deploy_calib_target_available_max': str(deploy_calib_meta['target_available'].max()),
        'dev_last_origin': str(dev_last_origin),
        'policy_ready_time': str(policy_ready_time),
        'earliest_holdout_origin': str(earliest_holdout_origin),
        'planned_eval_start': str(planned_eval_start),
        'planned_eval_end': str(planned_eval_end)
    },
    'validation_summary': summary_table.to_dict(orient='records'),
    'df_folds': df_folds.to_dict(orient='records'),
    'diagnostics': diagnostics,
    'winner': {
        'candidate': winner_cand,
        'budget': winner_budget,
        'aggregate_coverage': winner_cov,
        'aggregate_workload': winner_wl,
        'target_achieved': bool(target_70_achieved)
    },
    'deploy_models': deploy_fitted_models,
    'frozen_thresholds': deploy_initial_thresholds
}

joblib.dump(exp_artifact, 'experiment.joblib')
print('Serialized experiment.joblib successfully saved on disk.')

=== SUMMARY TABLE (All Candidates & Budgets) ===
                          candidate  budget  covered_fail  total_fail  agg_coverage  covered_n  total_n  agg_workload
                  baseline_rate168h    0.20             5          35      0.142857        125      879      0.142207
                  baseline_rate168h    0.25             5          35      0.142857        153      879      0.174061
                  baseline_rate168h    0.30             5          35      0.142857        153      879      0.174061
                   baseline_rate24h    0.20             3          35      0.085714         70      879      0.079636
                   baseline_rate24h    0.25             5          35      0.142857         92      879      0.104664
                   baseline_rate24h    0.30             5          35      0.142857         92      879      0.104664
                   baseline_rate72h    0.20             3          35      0.085714        110      879      0.125142
       

D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [ 16  24  53  60  63  80  85 108 152 160 189 190 197 200 201 202 203 204
 205 217 220 237 240 241 242 243 244 245 246 247 248 251 252 253 254 267
 268 269 270 271 272 273 274 275 276 277 287 295 324 325 326 333 336 337
 338 339 340 341 353 358 375 380 381 382 383 384 385 386 389 390 391 392
 405 406 407 408 409 410 411 412 413 414 415 425 433 460 461 462 469 472
 473 474 475 476 477 489 492 509 512 513 514 515 516 517 518 519 520 523
 524 525 526 539 540 541 542 543 544 545 546 547 548 549] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
D:\AI_Bussiness_Huflit\Đồ án\Industry_4_AI\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw



=== FROZEN HOLDOUT EVALUATION PROTOCOL ===
Earliest Holdout Origin: 2008-10-03 00:00:00
Planned Eval Start: 2008-10-03 06:00:00
Planned Eval End: 2008-10-17 06:00:00
Serialized experiment.joblib successfully saved on disk.
